# Telemetry AD Results Dashboard

This is the single presentation notebook for the project. It uses the current report-backed files under `reports/<dataset>/<variant>/` and focuses on:

- short variant labels
- conclusion-first summary tables
- baseline and advanced model comparisons
- full-model heatmaps
- time-series plots for each evaluated variant

In [32]:
from pathlib import Path
import json

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

ROOT = Path('.').resolve()
if not (ROOT / 'reports').exists():
    ROOT = ROOT.parent
REPORTS = ROOT / 'reports'

MODELS = ['zscore', 'iforest', 'lstm_ae', 'cnn_ae']
BASELINE_MODELS = ['zscore', 'iforest']
ADVANCED_MODELS = ['lstm_ae', 'cnn_ae']
METRIC_COLUMNS = ['accuracy', 'precision', 'recall', 'f1', 'event_f1', 'pr_auc', 'roc_auc']

DISPLAY_LABELS = {
    ('nab', 'ec2_cpu_utilization_5f5533'): 'NAB EC2 CPU',
    ('nab', 'ec2_network_in_257a54'): 'NAB Network',
    ('nab', 'rds_cpu_utilization_cc0c53'): 'NAB RDS CPU',
    ('skab', 'anomalyfree_vs_valve1_1'): 'SKAB Valve1',
}
DISPLAY_ORDER = list(DISPLAY_LABELS.values())

sns.set_theme(style='whitegrid', context='talk', palette='Set2')


MODEL_LABELS = {
    'zscore': 'Z-Score',
    'iforest': 'Isolation Forest',
    'lstm_ae': 'LSTM AE',
    'cnn_ae': 'CNN AE',
}

TYPE_LABELS = {
    'point': 'Point',
    'contextual': 'Contextual',
    'collective': 'Collective',
}

INTERPRETATION_LABELS = {
    'possible_fault_or_short_transient': 'Fault / transient',
    'possible_context_dependent_abnormality': 'Context abnormality',
    'possible_performance_degradation_or_sustained_fault': 'Degradation / sustained fault',
    'possible_concept_drift_or_setpoint_shift': 'Drift / setpoint shift',
}


def _format_short_number(value):
    if value is None or pd.isna(value):
        return ''
    value = float(value)
    abs_value = abs(value)
    if abs_value >= 1_000_000:
        return f"{value / 1_000_000:.1f}M"
    if abs_value >= 1_000:
        return f"{value / 1_000:.1f}K"
    return f"{value:.2f}"


def _format_peak_ts(value):
    if not value:
        return ''
    text = str(value)
    return text.replace('+00:00', '')


def _short_rationale(text):
    mapping = {
        'Single-window anomaly is most consistent with a short transient or isolated fault-like spike.': 'Short isolated spike or transient.',
        'Short contextual anomaly suggests behavior that is unusual relative to nearby context and could resemble attack-like or mode-specific activity.': 'Short context-dependent abnormality.',
        'Long collective anomaly that persists to the end of the evaluated horizon is treated as drift-like or setpoint-shift-like behavior.': 'Long event persisting to the end, treated as drift-like.',
        'Collective anomaly over multiple consecutive windows is most consistent with degradation or a sustained fault condition.': 'Sustained anomaly, consistent with degradation or fault.',
    }
    return mapping.get(str(text), str(text))


def _style_display(df, wrap_cols=None):
    wrap_cols = wrap_cols or []
    styler = df.style.hide(axis='index')
    styler = styler.set_table_styles([
        {'selector': 'th', 'props': [('text-align', 'left')]},
        {'selector': 'td', 'props': [('text-align', 'left'), ('vertical-align', 'top')]},
    ])
    if wrap_cols:
        styler = styler.set_properties(subset=wrap_cols, **{'white-space': 'normal', 'max-width': '320px'})
    return styler


def collect_metrics(models=MODELS):
    rows = []
    for metrics_path in sorted(REPORTS.glob('*/*/metrics.json')):
        payload = json.loads(metrics_path.read_text(encoding='utf-8'))
        dataset = payload['dataset']
        variant = payload['variant']
        display_label = DISPLAY_LABELS.get((dataset, variant), f'{dataset}:{variant}')
        for model in models:
            item = payload.get(model)
            if not item:
                continue
            cm = item.get('confusion_matrix', [[None, None], [None, None]])
            tn, fp = cm[0]
            fn, tp = cm[1]
            total = sum(v for v in [tn, fp, fn, tp] if v is not None)
            accuracy = ((tn + tp) / total) if total and None not in [tn, tp] else None
            rows.append({
                'dataset': dataset,
                'variant': variant,
                'display_label': display_label,
                'model': model,
                'accuracy': item.get('accuracy', accuracy),
                'precision': item.get('precision'),
                'recall': item.get('recall'),
                'f1': item.get('f1'),
                'event_f1': item.get('event_f1'),
                'pr_auc': item.get('pr_auc'),
                'roc_auc': item.get('roc_auc'),
                'threshold': item.get('threshold'),
                'pred_event_count': item.get('pred_event_count'),
            })
    df = pd.DataFrame(rows)
    numeric_cols = METRIC_COLUMNS + ['threshold', 'pred_event_count']
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df['model_family'] = df['model'].map(lambda m: 'baseline' if m in BASELINE_MODELS else 'advanced')
    return df.sort_values(['dataset', 'variant', 'model']).reset_index(drop=True)

def load_combined_predictions(dataset, variant):
    base = pd.read_csv(REPORTS / dataset / variant / 'predictions.csv')
    adv = pd.read_csv(REPORTS / dataset / variant / 'advanced_predictions.csv')
    for frame in (base, adv):
        if 'timestamp' in frame.columns:
            try:
                frame['timestamp'] = pd.to_datetime(frame['timestamp'])
            except Exception:
                pass
    return base.merge(adv, on=['timestamp', 'y_true'], how='outer').reset_index(drop=True)

def _label_order(source_df):
    present = list(source_df['display_label'].drop_duplicates())
    return [label for label in DISPLAY_ORDER if label in present]


def comparison_barplot(source_df, models, metric, title):
    subset = source_df[source_df['model'].isin(models)].copy()
    plt.figure(figsize=(11, 5))
    ax = sns.barplot(data=subset, x='display_label', y=metric, hue='model', order=_label_order(subset))
    ax.set_title(title)
    ax.set_xlabel('Dataset / Variant')
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=15)
    plt.tight_layout()
    plt.show()



def load_report_json(dataset, variant, filename):
    report_path = REPORTS / dataset / variant / filename
    if not report_path.exists():
        return []
    return json.loads(report_path.read_text(encoding='utf-8'))


def _summarize_explanation(entry):
    if 'top_features' in entry:
        top = entry.get('top_features', [])[:2]
        parts = [f"{item.get('feature')} ({item.get('zscore', 0):.1f})" for item in top]
        return '; '.join(parts)
    if 'top_extreme_features' in entry:
        top = entry.get('top_extreme_features', [])[:2]
        parts = [f"{item.get('feature')} ({item.get('standardized_magnitude', 0):.1f})" for item in top]
        return '; '.join(parts)
    if 'top_channels' in entry:
        top = entry.get('top_channels', [])[:2]
        parts = [f"{item.get('feature')} ({item.get('channel_error', 0):.1f})" for item in top]
        return '; '.join(parts)
    if 'top_timesteps' in entry:
        top = entry.get('top_timesteps', [])[:2]
        parts = [f"{item.get('relative_label')} ({item.get('mse', 0):.1f})" for item in top]
        return '; '.join(parts)
    return ''


def build_explainability_examples(best_rows):
    rows = []
    for item in best_rows.to_dict(orient='records'):
        dataset = item['dataset']
        variant = item['variant']
        model = item['model']
        display_label = item['display_label']
        explanations = load_report_json(dataset, variant, f'explanations_{model}.json')
        if not explanations:
            continue
        example = max(explanations, key=lambda e: float(e.get('peak_score', 0.0)))
        rows.append({
            'Dataset': display_label,
            'Model': MODEL_LABELS.get(model, model),
            'Type': TYPE_LABELS.get(example.get('type'), example.get('type')),
            'Peak time': _format_peak_ts(example.get('peak_ts')),
            'Peak score': _format_short_number(example.get('peak_score')),
            'Top contributors': _summarize_explanation(example),
        })
    return pd.DataFrame(rows)


def build_interpretation_examples(best_rows):
    rows = []
    for item in best_rows.to_dict(orient='records'):
        dataset = item['dataset']
        variant = item['variant']
        model = item['model']
        display_label = item['display_label']
        interpreted = load_report_json(dataset, variant, f'interpreted_events_{model}.json')
        if not interpreted:
            continue
        example = max(interpreted, key=lambda e: int(e.get('length', 0)))
        rows.append({
            'Dataset': display_label,
            'Model': MODEL_LABELS.get(model, model),
            'Type': TYPE_LABELS.get(example.get('type'), example.get('type')),
            'Length': int(example.get('length', 0)),
            'Interpretation': INTERPRETATION_LABELS.get(example.get('operational_interpretation'), example.get('operational_interpretation')),
            'Why': _short_rationale(example.get('interpretation_rationale')),
        })
    return pd.DataFrame(rows)


def build_event_count_summary(best_rows):
    rows = []
    for item in best_rows.to_dict(orient='records'):
        dataset = item['dataset']
        variant = item['variant']
        model = item['model']
        display_label = item['display_label']
        events = load_report_json(dataset, variant, f'events_{model}.json')
        interpreted = load_report_json(dataset, variant, f'interpreted_events_{model}.json')
        type_counts = {'point': 0, 'contextual': 0, 'collective': 0}
        for event in events:
            key = event.get('type', 'point')
            type_counts[key] = type_counts.get(key, 0) + 1
        op_counts = {}
        for event in interpreted:
            key = event.get('operational_interpretation', '')
            if key:
                op_counts[key] = op_counts.get(key, 0) + 1
        dominant = max(op_counts, key=op_counts.get) if op_counts else None
        rows.append({
            'Dataset': display_label,
            'Model': MODEL_LABELS.get(model, model),
            'Point': type_counts.get('point', 0),
            'Context': type_counts.get('contextual', 0),
            'Collective': type_counts.get('collective', 0),
            'Dominant label': INTERPRETATION_LABELS.get(dominant, dominant),
        })
    return pd.DataFrame(rows)

def heatmap_for_metric(source_df, metric, title):
    pivot = source_df.pivot(index='display_label', columns='model', values=metric).reindex(_label_order(source_df))
    plt.figure(figsize=(8.5, 4.5))
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlGnBu')
    plt.title(title)
    plt.xlabel('Model')
    plt.ylabel('Dataset / Variant')
    plt.tight_layout()
    plt.show()

def plot_variant_timeseries(dataset, variant, max_points=900):
    pred = load_combined_predictions(dataset, variant).copy()
    if len(pred) > max_points:
        pred = pred.iloc[-max_points:].reset_index(drop=True)
    x = pred['timestamp'] if 'timestamp' in pred.columns else pred.index
    display_label = DISPLAY_LABELS.get((dataset, variant), f'{dataset}:{variant}')
    specs = [
        ('zscore', 'tab:blue'),
        ('iforest', 'tab:orange'),
        ('lstm_ae', 'tab:green'),
        ('cnn_ae', 'tab:red'),
    ]
    fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
    for model, color in specs:
        score_col = f'{model}_score'
        pred_col = f'{model}_pred'
        if score_col in pred.columns:
            target_ax = axes[0] if model in BASELINE_MODELS else axes[1]
            target_ax.plot(x, pred[score_col], label=model, color=color, linewidth=1.0)
        if pred_col in pred.columns:
            idx = pred.index[pred[pred_col] == 1]
            if len(idx):
                axes[2].scatter(x.iloc[idx] if hasattr(x, 'iloc') else idx, [model] * len(idx), s=10, color=color, label=f'{model}_pred')
    if 'y_true' in pred.columns:
        idx_true = pred.index[pred['y_true'] == 1]
        if len(idx_true):
            axes[2].scatter(x.iloc[idx_true] if hasattr(x, 'iloc') else idx_true, ['ground_truth'] * len(idx_true), s=12, color='black', marker='x', label='ground_truth')
    axes[0].set_title(f'Baseline scores: {display_label}')
    axes[0].set_ylabel('score')
    axes[0].legend(loc='upper right')
    axes[1].set_title('Advanced-model scores')
    axes[1].set_ylabel('score')
    axes[1].legend(loc='upper right')
    axes[2].set_title('Predicted anomalies vs ground truth')
    axes[2].set_ylabel('events')
    axes[2].legend(loc='upper right')
    plt.tight_layout()
    plt.show()

df = collect_metrics()
variant_guide = pd.DataFrame([
    {'display_label': DISPLAY_LABELS[(dataset, variant)], 'dataset': dataset.upper(), 'variant': variant}
    for dataset, variant in DISPLAY_LABELS
])
print('ROOT:', ROOT)
print('Variants loaded:', len(df[['dataset', 'variant']].drop_duplicates()))


ROOT: C:\Users\orest\Desktop\University\Master AI\MAI650_IoT\Project\telemetry-ad
Variants loaded: 4


## Variant Guide

The plots use short labels to stay readable. This table maps each label back to the full dataset and variant.

In [33]:
display(variant_guide)

,display_label,dataset,variant
0,NAB EC2 CPU,NAB,ec2_cpu_utilization_5f5533
1,NAB Network,NAB,ec2_network_in_257a54
2,NAB RDS CPU,NAB,rds_cpu_utilization_cc0c53
3,SKAB Valve1,SKAB,anomalyfree_vs_valve1_1


## Variant Guide

The plots use short labels to stay readable. This table maps each label back to the full dataset and variant.

In [ ]:
display(variant_guide)

## Best By Dataset

In [ ]:
best_per_variant = df.loc[df.groupby(['dataset', 'variant'])['f1'].idxmax()].sort_values(['dataset', 'variant'])
best_per_dataset = df.loc[df.groupby('dataset')['f1'].idxmax()].sort_values('dataset')
display(best_per_dataset[['dataset', 'display_label', 'model', 'f1', 'event_f1', 'precision', 'recall']].round(4))


## NAB Focus

In [ ]:
nab_df = df[df['dataset'] == 'nab'].copy()
nab_best = best_per_variant[best_per_variant['dataset'] == 'nab'].copy()
display(nab_best[['display_label', 'model', 'f1', 'event_f1', 'precision', 'recall', 'pred_event_count']].round(4))

nab_baseline = nab_df[nab_df['model'].isin(BASELINE_MODELS)].copy()
display(nab_baseline[['display_label', 'model', 'f1', 'event_f1', 'accuracy', 'precision', 'recall']].round(4))
comparison_barplot(nab_baseline, BASELINE_MODELS, 'f1', 'NAB Baseline: Point-level F1')
comparison_barplot(nab_baseline, BASELINE_MODELS, 'event_f1', 'NAB Baseline: Event-level F1')
heatmap_for_metric(nab_baseline, 'f1', 'NAB Baseline F1 Heatmap')

nab_advanced = nab_df[nab_df['model'].isin(ADVANCED_MODELS)].copy()
display(nab_advanced[['display_label', 'model', 'f1', 'event_f1', 'accuracy', 'precision', 'recall']].round(4))
comparison_barplot(nab_advanced, ADVANCED_MODELS, 'f1', 'NAB Advanced: Point-level F1')
comparison_barplot(nab_advanced, ADVANCED_MODELS, 'event_f1', 'NAB Advanced: Event-level F1')
heatmap_for_metric(nab_advanced, 'f1', 'NAB Advanced F1 Heatmap')

display(nab_df.groupby('model')[METRIC_COLUMNS].mean().round(4))
comparison_barplot(nab_df, MODELS, 'f1', 'NAB All Models: Point-level F1')
comparison_barplot(nab_df, MODELS, 'event_f1', 'NAB All Models: Event-level F1')
heatmap_for_metric(nab_df, 'f1', 'NAB All Models F1 Heatmap')
heatmap_for_metric(nab_df, 'pr_auc', 'NAB All Models PR-AUC Heatmap')

nab_explainability = build_explainability_examples(nab_best)
nab_interpretation = build_interpretation_examples(nab_best)
nab_event_summary = build_event_count_summary(nab_best)
display(_style_display(nab_explainability, wrap_cols=['Top contributors']))
display(_style_display(nab_interpretation, wrap_cols=['Interpretation', 'Why']))
display(_style_display(nab_event_summary, wrap_cols=['Dominant label']))

display(nab_df[['display_label', 'model', 'accuracy', 'precision', 'recall', 'f1', 'event_f1', 'pr_auc', 'roc_auc', 'threshold', 'pred_event_count']].round(4))


## SKAB Focus

In [ ]:
skab_df = df[df['dataset'] == 'skab'].copy()
skab_best = best_per_variant[best_per_variant['dataset'] == 'skab'].copy()
display(skab_best[['display_label', 'model', 'f1', 'event_f1', 'precision', 'recall', 'pred_event_count']].round(4))

skab_baseline = skab_df[skab_df['model'].isin(BASELINE_MODELS)].copy()
display(skab_baseline[['display_label', 'model', 'f1', 'event_f1', 'accuracy', 'precision', 'recall']].round(4))
comparison_barplot(skab_baseline, BASELINE_MODELS, 'f1', 'SKAB Baseline: Point-level F1')
comparison_barplot(skab_baseline, BASELINE_MODELS, 'event_f1', 'SKAB Baseline: Event-level F1')
heatmap_for_metric(skab_baseline, 'f1', 'SKAB Baseline F1 Heatmap')

skab_advanced = skab_df[skab_df['model'].isin(ADVANCED_MODELS)].copy()
display(skab_advanced[['display_label', 'model', 'f1', 'event_f1', 'accuracy', 'precision', 'recall']].round(4))
comparison_barplot(skab_advanced, ADVANCED_MODELS, 'f1', 'SKAB Advanced: Point-level F1')
comparison_barplot(skab_advanced, ADVANCED_MODELS, 'event_f1', 'SKAB Advanced: Event-level F1')
heatmap_for_metric(skab_advanced, 'f1', 'SKAB Advanced F1 Heatmap')

display(skab_df.groupby('model')[METRIC_COLUMNS].mean().round(4))
comparison_barplot(skab_df, MODELS, 'f1', 'SKAB All Models: Point-level F1')
comparison_barplot(skab_df, MODELS, 'event_f1', 'SKAB All Models: Event-level F1')
heatmap_for_metric(skab_df, 'f1', 'SKAB All Models F1 Heatmap')
heatmap_for_metric(skab_df, 'pr_auc', 'SKAB All Models PR-AUC Heatmap')

skab_explainability = build_explainability_examples(skab_best)
skab_interpretation = build_interpretation_examples(skab_best)
skab_event_summary = build_event_count_summary(skab_best)
display(_style_display(skab_explainability, wrap_cols=['Top contributors']))
display(_style_display(skab_interpretation, wrap_cols=['Interpretation', 'Why']))
display(_style_display(skab_event_summary, wrap_cols=['Dominant label']))

display(skab_df[['display_label', 'model', 'accuracy', 'precision', 'recall', 'f1', 'event_f1', 'pr_auc', 'roc_auc', 'threshold', 'pred_event_count']].round(4))


## Time-Series Plots

In [ ]:
for dataset, variant in df[['dataset', 'variant']].drop_duplicates().itertuples(index=False):
    plot_variant_timeseries(dataset, variant, max_points=900)
